# 意味で検索する（TF-IDF vs 埋め込み）

実践編（総合演習）： Yuta Kimura / 木村祐太 の学習コース

同じ文書集合に対して、キーワードの一致で探す検索（**TF-IDF**）と、意味の近さで探す検索（**文埋め込み**）を両方作り、同じ質問で結果を並べて比べます。二つの検索は、最後は同じ **コサイン類似度で近いものを選ぶ** という形に落ち着き、違うのは文書を数値にする方法だけです。

このノートブックは Colab 前提です。上から順に実行してください。埋め込みモデルは初回に自動でダウンロードされます。

In [ ]:
!pip install scikit-learn sentence-transformers

## Step 1: データ入手：文書集合を用意する

実在の公開コーパス **20 Newsgroups** を使います。話題の異なる4カテゴリ（宇宙・自動車・コンピュータグラフィックス・野球）に絞り、ヘッダや署名・引用を落として本文だけにします。まずは先頭200件を使います。

In [ ]:
from sklearn.datasets import fetch_20newsgroups

cats = ["sci.space", "rec.autos",
        "comp.graphics", "rec.sport.baseball"]

data = fetch_20newsgroups(
    subset="train", categories=cats,
    remove=("headers", "footers", "quotes"),
    random_state=42)

docs = data.data[:200]
print("documents:", len(docs))
print(docs[0][:160])

## Step 2: 前処理：テキストをそろえる

TF-IDF は語の一致を見るので、小文字化し、英数字以外を空白に置き換えて語を切り出しやすくします。空や極端に短い文書は落とします。なお Step 4 の埋め込みには **整形前の元の文** を渡します（大文字小文字や句読点も手がかりになるため）。

In [ ]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # 英数字以外を空白に
    text = re.sub(r"\s+", " ", text).strip()  # 連続する空白をまとめる
    return text

clean = [normalize(d) for d in docs]
# 短すぎる文書は検索の役に立たないので除く
clean = [d for d in clean if len(d.split()) >= 5]

print("kept:", len(clean))
print(clean[0][:120])

## Step 3: TF-IDF 検索：語の重みで探す

**TF-IDF** は、文書中での語の頻度（TF）を、コーパス全体でのめずらしさ（IDF）で重み付けした特徴量です。各文書と質問を同じ空間のベクトルにし、**コサイン類似度**（ベクトルの向きの近さ、0 から 1）が高い順に並べます。

`TfidfVectorizer` がベクトル化を、`cosine_similarity` が類似度計算を担います。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words="english", min_df=2)
doc_matrix = vectorizer.fit_transform(clean)
print("tf-idf matrix:", doc_matrix.shape)  # (文書数, 語彙数)

def search_tfidf(query, k=3):
    q = vectorizer.transform([normalize(query)])
    sims = cosine_similarity(q, doc_matrix).ravel()
    top = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top]

for i, score in search_tfidf("rocket launch into orbit"):
    print(f"{score:.3f}  {clean[i][:64]}")

## Step 4: 意味検索：文埋め込みで探す

**文埋め込み**は、文全体を意味を表す固定長のベクトルに変換したものです。`sentence-transformers` の実在モデル **all-MiniLM-L6-v2** を使うと、各文が384次元のベクトルになります。語が違っても意味が近い文どうしは近くに配置されるので、言い換えを拾えます。

検索の仕組みは TF-IDF と同じで、**変わるのは文書を数値にする方法だけ** です。

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# 埋め込みには整形前の元の文を渡す
doc_emb = model.encode(docs, batch_size=32, show_progress_bar=True)
print("embeddings:", doc_emb.shape)  # (文書数, 384)

def search_embed(query, k=3):
    q = model.encode([query])
    sims = cosine_similarity(q, doc_emb).ravel()
    top = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top]

for i, score in search_embed("a spacecraft heading into orbit"):
    print(f"{score:.3f}  {docs[i][:64].strip()}")

## Step 5: 比較：言い換えで差が出る

小さく読みやすい文集合に、**本文の語と重ならない言い換えの質問** を投げます。質問は「a vehicle that travels through space（宇宙を移動する乗り物）」。TF-IDF は語が一致せず取りこぼし、埋め込みは意味で拾えるはずです。

In [ ]:
mini = [
    "The rocket lifted off and reached orbit.",
    "Astronauts prepared the spacecraft for launch.",
    "A sports car with a powerful engine was unveiled.",
    "The team won the baseball championship last night.",
]
query = "a vehicle that travels through space"

# TF-IDF 側
tv = TfidfVectorizer(stop_words="english")
M = tv.fit_transform(mini)
tfidf_sims = cosine_similarity(tv.transform([query]), M).ravel()

# 埋め込み側
emb = model.encode(mini)
embed_sims = cosine_similarity(model.encode([query]), emb).ravel()

print("tfidf  embed  sentence")
for i, s in enumerate(mini):
    print(f"{tfidf_sims[i]:.3f}  {embed_sims[i]:.3f}  {s}")

## Step 6: 考察：コストと使い分け

- **TF-IDF**：語の統計だけで軽く作れ、完全一致（型番・固有名詞・識別子）に強く、スコアの根拠を語で説明しやすい。
- **埋め込み**：学習済みモデルが要り計算は重いが、言い換えや同義語に強い。意味が近いだけの無関係な文を拾うことがある。
- 件数が増えると、全文書とのコサイン類似度を計算する総当りは遅くなる。実運用では **近似最近傍（FAISS など）** で高速化するのが定石。

**発展課題**：別のコーパス／別の埋め込みモデル（`all-mpnet-base-v2` など）／FAISS による近似最近傍／TF-IDF と埋め込みのスコアを足し合わせるハイブリッド検索。